# Mutual Information Feature Selection Practice

**Topic:** Information Theory &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(3000, 40)`, `y_train`: binary labels. Exactly 5 features carry signal, and several of them are **nonlinear and non-monotone** (strong in MI, nearly invisible to Pearson correlation). The other 35 are noise.
- `X_test` `(1000, 40)`: predict these, and `y_test` is used by the evaluation cell.

## Your task

1. Estimate `I(X_j; Y)` for every feature: bin each feature into ~12 quantile bins and compute `H(Y) − H(Y | bin)` from count tables (or use `sklearn.feature_selection.mutual_info_classif` to check which features rank highest. It uses a nearest-neighbour estimator, so its values differ from your binned ones).
2. Keep the top ~8 features by MI.
3. Train any nonlinear classifier on the selected features and set `predictions` for `X_test`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

d = 40

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d))
    score = (
        1.8 * (np.abs(X[:, 3]) < 0.7)          # U-shaped: near-zero correlation
        + 1.2 * np.tanh(2 * X[:, 11])          # saturating
        + 1.0 * (X[:, 17] > 1.2)               # threshold
        + 0.9 * X[:, 24]                       # plain linear
        + 1.1 * (X[:, 31] ** 2 - 1)            # quadratic: near-zero correlation
    )
    y = (score + r.normal(0, 0.6, n) > 0.9).astype(int)
    return X, y

X_train, y_train = make(3000, 271)
X_test, y_test = make(1000, 272)
print(f"X_train {X_train.shape}: only features 3, 11, 17, 24, 31 matter")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Bin each feature into 12 quantile bins and compute `I(X_j; Y) = H(Y) - H(Y | bin)` from count tables. This catches the U-shaped and quadratic features that Pearson correlation misses. Train a random forest on the top-8 MI features.

```python
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def mutual_info(x, y, bins=12):
    edges = np.quantile(x, np.linspace(0, 1, bins + 1))
    b = np.clip(np.digitize(x, edges[1:-1]), 0, bins - 1)
    py = np.bincount(y) / len(y)
    H_y = -np.sum(py * np.log(py + 1e-12))
    H_cond = 0.0
    for k in range(bins):
        m = b == k
        if not m.any():
            continue
        pk = np.bincount(y[m], minlength=2) / m.sum()
        H_cond += m.mean() * -np.sum(pk * np.log(pk + 1e-12))
    return H_y - H_cond

mi = np.array([mutual_info(X_train[:, j], y_train) for j in range(X_train.shape[1])])
top = np.argsort(mi)[::-1][:8]
print("selected features:", sorted(top.tolist()))

clf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
clf.fit(X_train[:, top], y_train)
predictions = clf.predict(X_test[:, top])
```
